# 14 — Decentralised coalition shielding

This notebook uses the same Chicken property,
$\mathbf{G}\neg\mathit{crash}$, but coalition members now choose their actions
**independently and simultaneously**.

The agents share the safety promise and may rely on teammates obeying their local
shield masks. They cannot observe or coordinate on teammates' current action
choices. Any agent outside the coalition would still be unrestricted.

The local masks must satisfy

$$
\prod_{i\in C} M_i(q,s)\subseteq A_C^{\mathrm{safe}}(q,s).
$$

A tuple that is safe only through runtime coordination cannot be exposed as
independent local choices.

In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import sys

import numpy as np

# Run from the repository root or from notebooks/tutorials/.
HERE = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (HERE, *HERE.parents) if (candidate / "masa").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the MASA-Safe-RL source tree.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from masa.common.multi_agent import Coalition
from masa.deterministic_shield import CoalitionLTLShield, random_safe
from masa.envs.multiagent.matrix.chicken import Actions
from masa.examples.chicken_safety_game import (
    make_labelled_chicken_env,
    make_never_crash_dfa,
)

ACTION_NAMES = {
    int(Actions.Swerve): "swerve",
    int(Actions.Straight): "straight",
}


def action_name(action: int) -> str:
    return ACTION_NAMES[int(action)]


def make_shield(
    coalition,
    *,
    mode="preemptive",
    execution="centralised",
    replacement=None,
    max_moves=8,
):
    return CoalitionLTLShield(
        make_labelled_chicken_env(max_moves=max_moves),
        coalition=Coalition(tuple(coalition)),
        dfa=make_never_crash_dfa(),
        mode=mode,
        execution=execution,
        replacement=replacement,
    )

## The central relation is not a set of local masks

For the full two-player coalition, centralised execution permits every pair except
`(straight, straight)`. Projecting this relation independently gives both players
both actions, recreating the unsafe pair.

In [ ]:
central = make_shield(
    ("player_0", "player_1"),
    mode="preemptive",
    execution="centralised",
)
central.reset(seed=0)
central_relation = central.coalition_action_mask()

print("Central relation:")
for joint, allowed in zip(central.coalition_actions, central_relation):
    print(tuple(action_name(action) for action in joint), bool(allowed))

naive_0 = {
    joint[0]
    for joint, allowed in zip(central.coalition_actions, central_relation)
    if allowed
}
naive_1 = {
    joint[1]
    for joint, allowed in zip(central.coalition_actions, central_relation)
    if allowed
}
print("Naïve marginal masks:", naive_0, naive_1)
assert (int(Actions.Straight), int(Actions.Straight)) in set(
    product(naive_0, naive_1)
)
assert not central_relation[
    central.encode_coalition_action((Actions.Straight, Actions.Straight))
]
central.close()

## Certified Cartesian local masks

MASA selects one deterministic Cartesian subset of the robust relation. Chicken
has multiple equally permissive rectangles. The canonical tie-break gives
`player_0` both choices and requires `player_1` to swerve. This asymmetry is a
pre-agreed safety interface—not runtime coordination.

In [ ]:
decentralised = make_shield(
    ("player_0", "player_1"),
    mode="preemptive",
    execution="decentralised",
)
_, infos = decentralised.reset(seed=0)

masks = decentralised.local_action_masks()
for agent, mask in masks.items():
    print(agent, [action_name(action) for action in np.flatnonzero(mask)])
print("Executable independent tuples:", decentralised.safe_coalition_actions())

# Exhaustively certify the Cartesian-product property.
for joint in product(
    *(np.flatnonzero(masks[agent]) for agent in decentralised.coalition_agents)
):
    index = decentralised.encode_coalition_action(tuple(map(int, joint)))
    assert decentralised.robust_coalition_action_mask()[index]

assert masks["player_0"].tolist() == [True, True]
assert masks["player_1"].tolist() == [True, False]

## Preemptive independent execution

Each local choice is checked against that agent's fixed local mask. A rejected
proposal does not advance the environment. The shield does not inspect one
teammate's proposal to decide what another teammate may do.

In [ ]:
try:
    decentralised.step(
        {
            "player_0": int(Actions.Swerve),
            "player_1": int(Actions.Straight),
        }
    )
except ValueError as exc:
    print("Rejected before stepping:", exc)

# player_0 may go straight because player_1's certified promise is to swerve.
_, _, _, _, infos = decentralised.step(
    {
        "player_0": int(Actions.Straight),
        "player_1": int(Actions.Swerve),
    }
)
assert "crash" not in infos["player_0"]["labels"]
print(
    "Executed:",
    {
        agent: action_name(infos[agent]["shield_executed_action"])
        for agent in decentralised.coalition_agents
    },
)
decentralised.close()

## Postposed replacement remains local

For decentralised postposed execution, each agent gets a distinct replacement
callback. It receives the shared product state, that agent's proposal, and that
agent's own mask—not teammates' current proposals.

In [ ]:
replacement = {
    "player_0": random_safe(seed=10),
    "player_1": random_safe(seed=11),
}
postposed = make_shield(
    ("player_0", "player_1"),
    mode="postposed",
    execution="decentralised",
    replacement=replacement,
)
postposed.reset(seed=0)

_, _, _, _, infos = postposed.step(
    {
        "player_0": int(Actions.Straight),
        "player_1": int(Actions.Straight),
    }
)
for agent in postposed.coalition_agents:
    print(
        agent,
        action_name(infos[agent]["shield_proposed_action"]),
        "->",
        action_name(infos[agent]["shield_executed_action"]),
        "intervened=",
        infos[agent]["shield_intervened"],
    )

assert infos["player_0"]["shield_executed_action"] == int(Actions.Straight)
assert infos["player_1"]["shield_executed_action"] == int(Actions.Swerve)
assert "crash" not in infos["player_0"]["labels"]
postposed.close()

## Proposal independence check

Repeat the same unsafe `player_1=straight` proposal while changing `player_0`'s
simultaneous proposal. `player_1` receives the same local result in both runs.

In [ ]:
executed_player_1 = []
for player_0_action in (Actions.Swerve, Actions.Straight):
    env = make_shield(
        ("player_0", "player_1"),
        mode="postposed",
        execution="decentralised",
    )
    env.reset(seed=0)
    _, _, _, _, infos = env.step(
        {
            "player_0": int(player_0_action),
            "player_1": int(Actions.Straight),
        }
    )
    executed_player_1.append(infos["player_1"]["shield_executed_action"])
    env.close()

print([action_name(action) for action in executed_player_1])
assert executed_player_1 == [int(Actions.Swerve), int(Actions.Swerve)]

## Takeaways

- Teammates are not adversaries: each may rely on the others obeying their
  certified local masks.
- Current actions remain uncoordinated, so the **entire Cartesian product** of
  those masks must be safe.
- Outsiders are different: all legal outsider actions are universally quantified
  during synthesis.
- The selected rectangle is sound and deterministic, but the greedy selector is
  not claimed to find a globally largest rectangle.
- Each member must identify the same global tabular state and DFA memory; this is
  not partial-observation synthesis.